In [0]:
CATALOG = "adwm_wh"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
TARGET_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dimproduct"

source_tables = [
    f"{CATALOG}.{SILVER_SCHEMA}.product",
    f"{CATALOG}.{SILVER_SCHEMA}.productmodel",
    f"{CATALOG}.{SILVER_SCHEMA}.productsubcategory",
    f"{CATALOG}.{SILVER_SCHEMA}.productcategory"
]

print(f"Target table: {TARGET_TABLE}")
print("Source tables:")
for table_name in source_tables:
    print(f"  - {table_name}")

In [0]:
%sql
WITH source_prepared AS (
    SELECT
        prod.ProductID AS ProductAlternateKey,
        sha2(
            concat_ws(
                '||',
                coalesce(cast(prod.ProductID AS STRING), '∅'),
                coalesce(cast(prod.Name AS STRING), '∅'),
                coalesce(cast(prod.ProductNumber AS STRING), '∅'),
                coalesce(cast(prod.ProductModelID AS STRING), '∅'),
                coalesce(cast(prod.ProductSubcategoryID AS STRING), '∅')
            ),
            256
        ) AS ProductVariantHash,
        prod.Name AS ProductName,
        prod.ProductNumber,
        prod.MakeFlag,
        prod.FinishedGoodsFlag,
        prod.Color,
        prod.Size,
        prod.SizeUnitMeasureCode,
        prod.Weight,
        prod.WeightUnitMeasureCode,
        prod.SafetyStockLevel,
        prod.ReorderPoint,
        prod.StandardCost,
        prod.ListPrice,
        prod.DaysToManufacture,
        prod.ProductLine,
        prod.Class,
        prod.Style,
        prod.SellStartDate,
        prod.SellEndDate,
        prod.DiscontinuedDate,
        prod.ProductModelID,
        pm.Name AS ProductModelName,
        pm.CatalogDescription,
        pm.Instructions,
        prod.ProductSubcategoryID,
        psc.Name AS ProductSubcategoryName,
        psc.ProductCategoryID,
        pc.Name AS ProductCategoryName,
        prod.rowguid AS RowGUID
    FROM adwm_wh.silver.product prod
    LEFT JOIN adwm_wh.silver.productmodel pm
        ON prod.ProductModelID = pm.ProductModelID
    LEFT JOIN adwm_wh.silver.productsubcategory psc
        ON prod.ProductSubcategoryID = psc.ProductSubcategoryID
    LEFT JOIN adwm_wh.silver.productcategory pc
        ON psc.ProductCategoryID = pc.ProductCategoryID
),
source_data AS (
    SELECT
        ProductAlternateKey,
        ProductVariantHash,
        ProductName,
        ProductNumber,
        MakeFlag,
        FinishedGoodsFlag,
        Color,
        Size,
        SizeUnitMeasureCode,
        Weight,
        WeightUnitMeasureCode,
        SafetyStockLevel,
        ReorderPoint,
        StandardCost,
        ListPrice,
        DaysToManufacture,
        ProductLine,
        Class,
        Style,
        SellStartDate,
        SellEndDate,
        DiscontinuedDate,
        ProductModelID,
        ProductModelName,
        CatalogDescription,
        Instructions,
        ProductSubcategoryID,
        ProductSubcategoryName,
        ProductCategoryID,
        ProductCategoryName,
        RowGUID,
        current_timestamp() AS ModifiedDate,
        sha2(
            concat_ws(
                '||',
                coalesce(cast(ProductAlternateKey AS STRING), '∅'),
                coalesce(cast(ProductVariantHash AS STRING), '∅'),
                coalesce(cast(ProductName AS STRING), '∅'),
                coalesce(cast(ProductNumber AS STRING), '∅'),
                coalesce(cast(MakeFlag AS STRING), '∅'),
                coalesce(cast(FinishedGoodsFlag AS STRING), '∅'),
                coalesce(cast(Color AS STRING), '∅'),
                coalesce(cast(Size AS STRING), '∅'),
                coalesce(cast(SizeUnitMeasureCode AS STRING), '∅'),
                coalesce(cast(Weight AS STRING), '∅'),
                coalesce(cast(WeightUnitMeasureCode AS STRING), '∅'),
                coalesce(cast(SafetyStockLevel AS STRING), '∅'),
                coalesce(cast(ReorderPoint AS STRING), '∅'),
                coalesce(cast(StandardCost AS STRING), '∅'),
                coalesce(cast(ListPrice AS STRING), '∅'),
                coalesce(cast(DaysToManufacture AS STRING), '∅'),
                coalesce(cast(ProductLine AS STRING), '∅'),
                coalesce(cast(Class AS STRING), '∅'),
                coalesce(cast(Style AS STRING), '∅'),
                coalesce(cast(SellStartDate AS STRING), '∅'),
                coalesce(cast(SellEndDate AS STRING), '∅'),
                coalesce(cast(DiscontinuedDate AS STRING), '∅'),
                coalesce(cast(ProductModelID AS STRING), '∅'),
                coalesce(cast(ProductModelName AS STRING), '∅'),
                coalesce(cast(CatalogDescription AS STRING), '∅'),
                coalesce(cast(Instructions AS STRING), '∅'),
                coalesce(cast(ProductSubcategoryID AS STRING), '∅'),
                coalesce(cast(ProductSubcategoryName AS STRING), '∅'),
                coalesce(cast(ProductCategoryID AS STRING), '∅'),
                coalesce(cast(ProductCategoryName AS STRING), '∅'),
                coalesce(cast(RowGUID AS STRING), '∅')
            ),
            256
        ) AS ChangeHash
    FROM source_prepared
)
MERGE INTO adwm_wh.gold.dimproduct AS target
USING source_data AS source
ON target.ProductVariantHash = source.ProductVariantHash
WHEN MATCHED AND source.ChangeHash <> sha2(
    concat_ws(
        '||',
        coalesce(cast(target.ProductAlternateKey AS STRING), '∅'),
        coalesce(cast(target.ProductVariantHash AS STRING), '∅'),
        coalesce(cast(target.ProductName AS STRING), '∅'),
        coalesce(cast(target.ProductNumber AS STRING), '∅'),
        coalesce(cast(target.MakeFlag AS STRING), '∅'),
        coalesce(cast(target.FinishedGoodsFlag AS STRING), '∅'),
        coalesce(cast(target.Color AS STRING), '∅'),
        coalesce(cast(target.Size AS STRING), '∅'),
        coalesce(cast(target.SizeUnitMeasureCode AS STRING), '∅'),
        coalesce(cast(target.Weight AS STRING), '∅'),
        coalesce(cast(target.WeightUnitMeasureCode AS STRING), '∅'),
        coalesce(cast(target.SafetyStockLevel AS STRING), '∅'),
        coalesce(cast(target.ReorderPoint AS STRING), '∅'),
        coalesce(cast(target.StandardCost AS STRING), '∅'),
        coalesce(cast(target.ListPrice AS STRING), '∅'),
        coalesce(cast(target.DaysToManufacture AS STRING), '∅'),
        coalesce(cast(target.ProductLine AS STRING), '∅'),
        coalesce(cast(target.Class AS STRING), '∅'),
        coalesce(cast(target.Style AS STRING), '∅'),
        coalesce(cast(target.SellStartDate AS STRING), '∅'),
        coalesce(cast(target.SellEndDate AS STRING), '∅'),
        coalesce(cast(target.DiscontinuedDate AS STRING), '∅'),
        coalesce(cast(target.ProductModelID AS STRING), '∅'),
        coalesce(cast(target.ProductModelName AS STRING), '∅'),
        coalesce(cast(target.CatalogDescription AS STRING), '∅'),
        coalesce(cast(target.Instructions AS STRING), '∅'),
        coalesce(cast(target.ProductSubcategoryID AS STRING), '∅'),
        coalesce(cast(target.ProductSubcategoryName AS STRING), '∅'),
        coalesce(cast(target.ProductCategoryID AS STRING), '∅'),
        coalesce(cast(target.ProductCategoryName AS STRING), '∅'),
        coalesce(cast(target.RowGUID AS STRING), '∅')
    ),
    256
) THEN UPDATE SET
    target.ProductAlternateKey = source.ProductAlternateKey,
    target.ProductName = source.ProductName,
    target.ProductNumber = source.ProductNumber,
    target.MakeFlag = source.MakeFlag,
    target.FinishedGoodsFlag = source.FinishedGoodsFlag,
    target.Color = source.Color,
    target.Size = source.Size,
    target.SizeUnitMeasureCode = source.SizeUnitMeasureCode,
    target.Weight = source.Weight,
    target.WeightUnitMeasureCode = source.WeightUnitMeasureCode,
    target.SafetyStockLevel = source.SafetyStockLevel,
    target.ReorderPoint = source.ReorderPoint,
    target.StandardCost = source.StandardCost,
    target.ListPrice = source.ListPrice,
    target.DaysToManufacture = source.DaysToManufacture,
    target.ProductLine = source.ProductLine,
    target.Class = source.Class,
    target.Style = source.Style,
    target.SellStartDate = source.SellStartDate,
    target.SellEndDate = source.SellEndDate,
    target.DiscontinuedDate = source.DiscontinuedDate,
    target.ProductModelID = source.ProductModelID,
    target.ProductModelName = source.ProductModelName,
    target.CatalogDescription = source.CatalogDescription,
    target.Instructions = source.Instructions,
    target.ProductSubcategoryID = source.ProductSubcategoryID,
    target.ProductSubcategoryName = source.ProductSubcategoryName,
    target.ProductCategoryID = source.ProductCategoryID,
    target.ProductCategoryName = source.ProductCategoryName,
    target.RowGUID = source.RowGUID,
    target.ModifiedDate = source.ModifiedDate
WHEN NOT MATCHED THEN INSERT (
    ProductAlternateKey,
    ProductVariantHash,
    ProductName,
    ProductNumber,
    MakeFlag,
    FinishedGoodsFlag,
    Color,
    Size,
    SizeUnitMeasureCode,
    Weight,
    WeightUnitMeasureCode,
    SafetyStockLevel,
    ReorderPoint,
    StandardCost,
    ListPrice,
    DaysToManufacture,
    ProductLine,
    Class,
    Style,
    SellStartDate,
    SellEndDate,
    DiscontinuedDate,
    ProductModelID,
    ProductModelName,
    CatalogDescription,
    Instructions,
    ProductSubcategoryID,
    ProductSubcategoryName,
    ProductCategoryID,
    ProductCategoryName,
    RowGUID,
    ModifiedDate
) VALUES (
    source.ProductAlternateKey,
    source.ProductVariantHash,
    source.ProductName,
    source.ProductNumber,
    source.MakeFlag,
    source.FinishedGoodsFlag,
    source.Color,
    source.Size,
    source.SizeUnitMeasureCode,
    source.Weight,
    source.WeightUnitMeasureCode,
    source.SafetyStockLevel,
    source.ReorderPoint,
    source.StandardCost,
    source.ListPrice,
    source.DaysToManufacture,
    source.ProductLine,
    source.Class,
    source.Style,
    source.SellStartDate,
    source.SellEndDate,
    source.DiscontinuedDate,
    source.ProductModelID,
    source.ProductModelName,
    source.CatalogDescription,
    source.Instructions,
    source.ProductSubcategoryID,
    source.ProductSubcategoryName,
    source.ProductCategoryID,
    source.ProductCategoryName,
    source.RowGUID,
    source.ModifiedDate
);

In [0]:
%sql
SELECT
    COUNT(*) AS dimproduct_count,
    COUNT(DISTINCT ProductVariantHash) AS distinct_productvarianthashes,
    COUNT(DISTINCT ProductAlternateKey) AS distinct_productalternatekeys
FROM adwm_wh.gold.dimproduct;